In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import librosa
import IPython.display as ipd
import librosa.display
from google.colab import drive
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import regularizers



In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# run this in every NEW session to prepare the data
# faster extraction compared to extracting inside google drive
!cp /content/drive/MyDrive/IRMAS-TrainingData.zip /content/
!unzip -q /content/IRMAS-TrainingData.zip -d /content/dataset

replace /content/dataset/IRMAS-TrainingData/cel/008__[cel][nod][cla]0058__1.wav? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/dataset/IRMAS-TrainingData/cel/008__[cel][nod][cla]0058__2.wav? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
# remove Readme.txt file
data_path = "dataset/IRMAS-TrainingData"
classes = sorted({
    d for d in os.listdir(data_path)
    if os.path.isdir(os.path.join(data_path, d))
})
classes

['cel', 'cla', 'flu', 'gac', 'gel', 'org', 'pia', 'sax', 'tru', 'vio', 'voi']

In [ ]:
len(classes)

11

In [ ]:
# load all files
X = []
y = []

for label, instrument in enumerate(classes):
  folder = os.path.join(data_path, instrument)

  for file in os.listdir(folder):
    file_path = os.path.join(folder, file)
    audio,sr = librosa.load(file_path, sr=44100)
    mel_spec = librosa.feature.melspectrogram(
        y=audio,
        sr=44100,
        n_mels=128,
        fmax=8000
    )
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    X.append(mel_spec_db)
    y.append(label)

In [ ]:
X = [spec.T for spec in X]
X = np.array(X)
y = np.array(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from numpy.random import shuffle
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(1000)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)


In [ ]:
class MusicPositionalEmbedding(keras.layers.Layer):
  def __init__(self, seq_length, output_dim, **kwargs):
    super().__init__(**kwargs)
    #instead of token ids, project the 128 mel bins into the embed_dim
    self.projection = keras.layers.Dense(output_dim)
    #position embedding
    self.position_embedding = keras.layers.Embedding(
        input_dim=seq_length,
        output_dim=output_dim
    )
    self.seq_length = seq_length
    self.output_dim = output_dim

  def call(self, inputs):
    length = tf.shape(inputs)[1]
    positions = tf.range(start=0, limit=length, delta=1)
    embedded_positions = self.position_embedding(positions)
    x = self.projection(inputs)
    return x + embedded_positions


In [ ]:
class TransformerEncoder(keras.layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [keras.layers.Dense(dense_dim, activation="relu"),
             keras.layers.Dense(embed_dim),]
        )
        self.layernorm_1 = keras.layers.LayerNormalization()
        self.layernorm_2 = keras.layers.LayerNormalization()

    def call(self, inputs, mask=None):
        if mask is not None:
            mask = mask[:, tf.newaxis, :]
        attention_output = self.attention(
            inputs, inputs, attention_mask=mask)
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dense_dim": self.dense_dim,
        })
        return config

In [ ]:
embed_dim = 128 #the musical word vector
dense_dim = 256 #hidden layer
num_heads = 8 #perspectives

model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1], 128)), # Time steps, 128 Mel-bins
    keras.layers.Normalization(name="norm_layer"),
    keras.layers.Conv1D(filters=embed_dim, kernel_size=3, padding="same", activation="relu"),
    MusicPositionalEmbedding(seq_length=X_train.shape[1], output_dim=embed_dim),
    TransformerEncoder(embed_dim, dense_dim, num_heads),
    TransformerEncoder(embed_dim, dense_dim, num_heads),
    #TransformerEncoder(embed_dim, dense_dim, num_heads),
    keras.layers.GlobalAveragePooling1D(), # Flatten the sequence into one summary
    keras.layers.Dropout(0.3),
    keras.layers.Dense(11, activation="softmax") # 11 instruments!
])
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy", # Standard version
    metrics=["accuracy"]
)
model.get_layer("norm_layer").adapt(X_train)
model.summary()

Model: "sequential_28"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ norm_layer (Normalization)      │ (None, 259, 128)       │           257 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 259, 128)       │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ music_positional_embedding_9    │ (None, 259, 128)       │        49,664 │
│ (MusicPositionalEmbedding)      │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_17          │ (None, 259, 128)       │       593,920 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_18          │ (None, 259, 128)       │       593,920 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_9      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_26 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_57 (Dense)                │ (None, 11)             │         1,419 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,288,460 (4.92 MB)

 Trainable params: 1,288,203 (4.91 MB)

 Non-trainable params: 257 (1.01 KB)

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="best_music_transformer.keras",
        save_best_only=True,
        monitor="val_loss"
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,  # Give it 5 epochs to try and improve before quitting
        restore_best_weights=True # After stopping, go back to the best version
    )
]

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50, # Set to 50, but EarlyStopping will likely stop it earlier
    callbacks=callbacks
)

Epoch 1/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 28s 108ms/step - accuracy: 0.2535 - loss: 2.3231 - val_accuracy: 0.3751 - val_loss: 1.8765
Epoch 2/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 10s 59ms/step - accuracy: 0.3762 - loss: 1.8781 - val_accuracy: 0.4198 - val_loss: 1.7227
Epoch 3/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 10s 61ms/step - accuracy: 0.4644 - loss: 1.6160 - val_accuracy: 0.4937 - val_loss: 1.5550
Epoch 4/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - accuracy: 0.5268 - loss: 1.4495 - val_accuracy: 0.5227 - val_loss: 1.4351
Epoch 5/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - accuracy: 0.5712 - loss: 1.3317 - val_accuracy: 0.5533 - val_loss: 1.4085
Epoch 6/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - accuracy: 0.5902 - loss: 1.2521 - val_accuracy: 0.5600 - val_loss: 1.3876
Epoch 7/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - accuracy: 0.6189 - loss: 1.1792 - val_accuracy: 0.5541 - val_loss: 1.4140
Epoch 8/50
168/168 ━━━━━━━━━━━━━━━━━━━━ 11s 64ms/step - accuracy: 0.6404 - loss: 1.1005 -